In [15]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)
import os

# ================= CONFIG =================
GROUND_TRUTH = "/content/ground_truth_classification_300(correct id) - ground_truth_classification_300.csv"

MODEL_FILES = {
    "GPT - 120": "/content/classified_articles_openai_gpt-oss-120b(300).csv",
    "Mistral": "/content/classified_articles_mistralai_Mixtral-8x7B-Instruct-v0.1(300).csv",
    "LLaMA": "/content/classified_articles_llama-3.3-70b-versatile(300).csv"
}

LIMIT_ROWS = None   # set to 50 if needed
ID_COL = "id"

# ==========================================

def find_column(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(f"Required column not found. Available: {df.columns.tolist()}")

# ---- Load Ground Truth ----
gt = pd.read_csv(GROUND_TRUTH)
if LIMIT_ROWS:
    gt = gt.head(LIMIT_ROWS)

gt_col = find_column(gt, ["ground_truth", "label", "class"])
gt[gt_col] = gt[gt_col].astype(str).str.strip()

results = []

# ---- Evaluate Models ----
for model, path in MODEL_FILES.items():
    try:
        if not os.path.exists(path) or os.path.getsize(path) < 10:
            print(f"⚠️ Skipping {model}: file missing or empty")
            continue

        df = pd.read_csv(path)
        if LIMIT_ROWS:
            df = df.head(LIMIT_ROWS)

        pred_col = find_column(df, ["classification", "prediction", "label"])
        df[pred_col] = df[pred_col].astype(str).str.strip()

        merged = pd.merge(gt, df, on=ID_COL, how="inner")

        if merged.empty:
            print(f"⚠️ No matching IDs for {model}")
            continue

        y_true = merged[gt_col]     # ✅ correct
        y_pred = merged[pred_col]   # ✅ correct

        metrics = {
            "Model": model,
            "Samples": len(merged),
            "Accuracy": accuracy_score(y_true, y_pred),

            "Precision_Macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
            "Recall_Macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
            "F1_Macro": f1_score(y_true, y_pred, average="macro", zero_division=0),

            "Precision_Weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
            "Recall_Weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
            "F1_Weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),

            "Precision_Micro": precision_score(y_true, y_pred, average="micro", zero_division=0),
            "Recall_Micro": recall_score(y_true, y_pred, average="micro", zero_division=0),
            "F1_Micro": f1_score(y_true, y_pred, average="micro", zero_division=0),
        }

        results.append(metrics)

        print(f"\n=== {model} CLASSIFICATION REPORT ===")
        print(classification_report(y_true, y_pred, digits=3))

    except Exception as e:
        print(f"❌ Error with {model}: {e}")

# ---- Final Table ----
summary = pd.DataFrame(results)
summary = summary.sort_values("F1_Macro", ascending=False)

out_path = "/content/model_comparison_metrics.csv"
summary.to_csv(out_path, index=False)

print("\n✅ FINAL MODEL COMPARISON (Sorted by Macro F1):")
print(summary)
print(f"\n📂 Saved to: {out_path}")



=== GPT CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

           1      0.945     0.984     0.964       243
           2      0.800     0.471     0.593        51
           3      0.118     0.333     0.174         6

    accuracy                          0.883       300
   macro avg      0.621     0.596     0.577       300
weighted avg      0.904     0.883     0.885       300


=== Mistral CLASSIFICATION REPORT ===
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

In [10]:
GROUND_TRUTH = "/content/ground_truth_classification_300(correct id) - ground_truth_classification_300.csv"

MODEL_FILES = {
    "GPT": "/content/classified_articles_openai_gpt-oss-120b(300).csv",
    "Mistral": "/content/classified_articles_mistralai_Mixtral-8x7B-Instruct-v0.1(300).csv",
    "LLaMA": "/content/classified_articles_llama-3.3-70b-versatile(300).csv"
}

LIMIT_ROWS = "none"   # set None to evaluate full dataset
ID_COL = "id"
GT_COL = "ground_truth"
PRED_COL = "classification"
GROUND_TRUTH_PATH = "/content/ground_truth_classification_300(correct id) - ground_truth_classification_300.csv"
PREDICTION_PATH   = "/content/classified_articles_llama-3.3-70b-versatile(300).csv"

In [11]:
gt = pd.read_csv(GROUND_TRUTH)
# Check if LIMIT_ROWS is not None and is not the string "none" (case-insensitive)
# Only apply head() if a valid integer limit is provided.
if LIMIT_ROWS is not None and str(LIMIT_ROWS).strip().lower() != "none":
    try:
        limit = int(LIMIT_ROWS)
        gt = gt.head(limit)
    except ValueError:
        # If LIMIT_ROWS is a truthy string but not "none" and not convertible to int,
        # we'll ignore it and not apply the head() operation.
        pass

gt[GT_COL] = gt[GT_COL].astype(str).str.strip().str.lower()

results = []

In [13]:
for model_name, path in MODEL_FILES.items():
    try:
        df = pd.read_csv(path)
        # Apply LIMIT_ROWS logic for the prediction dataframe as well
        if LIMIT_ROWS is not None and str(LIMIT_ROWS).strip().lower() != "none":
            try:
                limit = int(LIMIT_ROWS)
                df = df.head(limit)
            except ValueError:
                pass # Ignore if LIMIT_ROWS is not a valid integer or "none"

        df[PRED_COL] = df[PRED_COL].astype(str).str.strip().str.lower()

        # Merge on ID
        merged = pd.merge(
            gt[[ID_COL, GT_COL]],
            df[[ID_COL, PRED_COL]],
            on=ID_COL,
            how="inner"
        )

        if merged.empty:
            print(f"⚠️ No matching IDs for {model_name}")
            continue

        y_true = merged[GT_COL]
        y_pred = merged[PRED_COL]

        # =========================
        # METRICS
        # =========================
        metrics = {
            "Accuracy": accuracy_score(y_true, y_pred),

            "Precision_Macro": precision_score(
                y_true, y_pred, average="macro", zero_division=0
            ),
            "Recall_Macro": recall_score(
                y_true, y_pred, average="macro", zero_division=0
            ),
            "F1_Macro": f1_score(
                y_true, y_pred, average="macro", zero_division=0
            ),

            "Precision_Weighted": precision_score(
                y_true, y_pred, average="weighted", zero_division=0
            ),
            "Recall_Weighted": recall_score(
                y_true, y_pred, average="weighted", zero_division=0
            ),
            "F1_Weighted": f1_score(
                y_true, y_pred, average="weighted", zero_division=0
            ),

            "Precision_Micro": precision_score(
                y_true, y_pred, average="micro", zero_division=0
            ),
            "Recall_Micro": recall_score(
                y_true, y_pred, average="micro", zero_division=0
            ),
            "F1_Micro": f1_score(
                y_true, y_pred, average="micro", zero_division=0
            ),
        }

        results.append({
            "Model": model_name,
            "Samples": len(merged),
            **{k: round(v, 4) for k, v in metrics.items()}
        })

    except Exception as e:
        print(f"An error occurred while processing {model_name}: {e}")
        continue


In [14]:
for model_name, path in MODEL_FILES.items():
    try:
        df = pd.read_csv(path)
        if LIMIT_ROWS:
            df = df.head(LIMIT_ROWS)

        df[PRED_COL] = df[PRED_COL].astype(str).str.strip().str.lower()

        # Merge on ID
        merged = pd.merge(
            gt[[ID_COL, GT_COL]],
            df[[ID_COL, PRED_COL]],
            on=ID_COL,
            how="inner"
        )

        if merged.empty:
            print(f"⚠️ No matching IDs for {model_name}")
            continue

        y_true = merged[GT_COL]
        y_pred = merged[PRED_COL]

        # =========================
        # METRICS
        # =========================
        metrics = {
            "Accuracy": accuracy_score(y_true, y_pred),

            "Precision_Macro": precision_score(
                y_true, y_pred, average="macro", zero_division=0
            ),
            "Recall_Macro": recall_score(
                y_true, y_pred, average="macro", zero_division=0
            ),
            "F1_Macro": f1_score(
                y_true, y_pred, average="macro", zero_division=0
            ),

            "Precision_Weighted": precision_score(
                y_true, y_pred, average="weighted", zero_division=0
            ),
            "Recall_Weighted": recall_score(
                y_true, y_pred, average="weighted", zero_division=0
            ),
            "F1_Weighted": f1_score(
                y_true, y_pred, average="weighted", zero_division=0
            ),

            "Precision_Micro": precision_score(
                y_true, y_pred, average="micro", zero_division=0
            ),
            "Recall_Micro": recall_score(
                y_true, y_pred, average="micro", zero_division=0
            ),
            "F1_Micro": f1_score(
                y_true, y_pred, average="micro", zero_division=0
            ),
        }

        results.append({
            "Model": model_name,
            "Samples": len(merged),
            **{k: round(v, 4) for k, v in metrics.items()}
        })

        # =========================
        # REPORT
        # =========================
        print(f"\n================ {model_name} =================")
        print(classification_report(y_true, y_pred, digits=3, zero_division=0))

    except Exception as e:
        print(f"❌ Error with {model_name}: {e}")

# =========================
# SAVE SUMMARY
# =========================
summary_df = pd.DataFrame(results)
summary_df = summary_df.sort_values("F1_Macro", ascending=False)

out_path = "/content/model_comparison_metrics.csv"
summary_df.to_csv(out_path, index=False)

print("\n✅ FINAL MODEL COMPARISON (Sorted by Macro F1):")
print(summary_df)
print(f"\n📂 Saved to: {out_path}")

❌ Error with GPT: cannot do positional indexing on RangeIndex with these indexers [none] of type str
❌ Error with Mistral: cannot do positional indexing on RangeIndex with these indexers [none] of type str
❌ Error with LLaMA: cannot do positional indexing on RangeIndex with these indexers [none] of type str

✅ FINAL MODEL COMPARISON (Sorted by Macro F1):
     Model  Samples  Accuracy  Precision_Macro  Recall_Macro  F1_Macro  \
0      GPT      300    0.8833           0.6208        0.5958    0.5767   
2    LLaMA      300    0.8200           0.4652        0.4175    0.3928   
1  Mistral      300    0.0067           0.0159        0.0001    0.0003   

   Precision_Weighted  Recall_Weighted  F1_Weighted  Precision_Micro  \
0              0.9035           0.8833       0.8848           0.8833   
2              0.7957           0.8200       0.7855           0.8200   
1              0.8100           0.0067       0.0132           0.0067   

   Recall_Micro  F1_Micro  
0        0.8833    0.8833  
2

In [9]:
import os

qwen_path = "/content/classified_articles_qwen_qwen3-32b(300).csv"

print("File exists:", os.path.exists(qwen_path))
print("File size (bytes):", os.path.getsize(qwen_path))

with open(qwen_path, "r", encoding="utf-8", errors="ignore") as f:
    print("\n--- FIRST 10 LINES ---")
    for i in range(10):
        print(f.readline())


File exists: True
File size (bytes): 1

--- FIRST 10 LINES ---











